In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
from statsmodels.tsa.statespace.structural import UnobservedComponents

#### Pivoting: adding missing days, fixing NaNs with zeros and sums

In [ ]:
def pivot_df(df_imputed, imputed_vars, name='median'):

    # ids and dates
    ids = df_imputed['id'].unique()
    df_imputed['date'] = pd.to_datetime(df_imputed['date']) 

    # summing
    sum_vars = [v for v in df_imputed['variable'].unique() if v not in imputed_vars]  
    median_group = df_imputed[df_imputed['variable'].isin(imputed_vars)].copy()
    median_agg = (median_group.groupby(['id', 'date', 'variable'])['value'].median().reset_index())

    sum_group = df_imputed[df_imputed['variable'].isin(sum_vars)].copy()
    sum_agg = (sum_group.groupby(['id', 'date', 'variable'])['value'].sum().reset_index())

    # pivot and combine
    combined = pd.concat([median_agg, sum_agg], ignore_index=True)
    daily_df = combined.pivot(index=['id', 'date'], columns='variable', values='value').reset_index()
    daily_df.columns.name = None

    # fill in 0 for sumtype variables
    all_columns = daily_df.columns.tolist()
    cols_to_fill = [col for col in all_columns if col not in imputed_vars and col not in ['id', 'date']]
    daily_df[cols_to_fill] = daily_df[cols_to_fill].fillna(0)
    print(f"{name} dataset: shape before adding missing dates = {daily_df.shape}")
    
    # missing dates per used added
    all_dates = pd.date_range(daily_df['date'].min(), daily_df['date'].max())
    full_index = pd.MultiIndex.from_product([ids, all_dates], names=['id', 'date'])
    daily_df_full = daily_df.set_index(['id', 'date']).reindex(full_index).reset_index()

    print(f"{name} dataset: shape after adding missing dates = {daily_df_full.shape}")
    display(daily_df_full.head())
    return daily_df, daily_df_full


In [ ]:
# datatsets from cleaning
df_median_imputed = pd.read_csv('df_median.csv')
df_kalman_imputed = pd.read_csv('df_kalman.csv')

# imputed variables
imputed_vars = ['mood', 'circumplex.arousal', 'circumplex.valence', 'activity']

# use function made previously
daily_df_median_raw, daily_df_median_full = pivot_df(df_median_imputed, imputed_vars, name='median')
daily_df_kalman_raw, daily_df_kalman_full = pivot_df(df_kalman_imputed, imputed_vars, name='kalman')

# to compare later shapes
print("Before trimming:")
print("median shape:", daily_df_median_full.shape)
print("kalman shape:", daily_df_kalman_full.shape)


In [ ]:
daily_df_median_raw[imputed_vars].isna().sum(), daily_df_kalman_raw[imputed_vars].isna().sum()


In [ ]:
daily_df_median_full[imputed_vars].isna().sum(), daily_df_kalman_full[imputed_vars].isna().sum()

#### Trimming the dataset till last known mood entry

In [ ]:
# trimming till last known mood for median method
if 'mood' in daily_df_median_full.columns:
    last_mood_dates = (
        daily_df_median_full[~daily_df_median_full['mood'].isna()]
        .groupby('id')['date'].max()
    )

    daily_df_median_full = daily_df_median_full.merge(
        last_mood_dates.rename('last_mood_date'), on='id', how='left'
    )
    daily_df_median_full = daily_df_median_full[daily_df_median_full['date'] <= daily_df_median_full['last_mood_date']]
    daily_df_median_full.drop(columns='last_mood_date', inplace=True)


# trimming till last known mood for kalman method
if 'mood' in daily_df_kalman_full.columns:
    last_mood_dates = (
        daily_df_kalman_full[~daily_df_kalman_full['mood'].isna()]
        .groupby('id')['date'].max()
    )

    daily_df_kalman_full = daily_df_kalman_full.merge(
        last_mood_dates.rename('last_mood_date'), on='id', how='left'
    )
    daily_df_kalman_full = daily_df_kalman_full[daily_df_kalman_full['date'] <= daily_df_kalman_full['last_mood_date']]
    daily_df_kalman_full.drop(columns='last_mood_date', inplace=True)

print("\nAfter trimming:")
print("median shape:", daily_df_median_full.shape)
print("kalman shape:", daily_df_kalman_full.shape)


#### Imputation: two methods again (median and kalman)

##### Method 1: Average medians

In [ ]:
# average median method
vars_to_impute = [col for col in daily_df_median_full.columns if col not in ['id', 'date']]

for var in vars_to_impute:
    # average of previous and next day
    prev = daily_df_median_full.groupby('id')[var].shift(1)
    next_ = daily_df_median_full.groupby('id')[var].shift(-1)
    interpolated = (prev + next_) / 2
    daily_df_median_full[var] = daily_df_median_full[var].fillna(interpolated)

    # forward and backward fill
    daily_df_median_full[var] = (
        daily_df_median_full.groupby('id')[var]
        .apply(lambda x: x.ffill().bfill())
        .reset_index(level=0, drop=True)
    )

daily_df_median_full.head()

In [ ]:
missing_values = daily_df_median_full.isna().sum()
missing_values

In [ ]:
# Define rules per variable
rules = {
    'mood': lambda x: (x < 1) | (x > 10),
    'circumplex.arousal': lambda x: (x < -2) | (x > 2),
    'circumplex.valence': lambda x: (x < -2) | (x > 2),
    'activity': lambda x: (x < 0) | (x > 1),
}

# Add all duration-like columns
duration_vars = [col for col in daily_df_median_full.columns if col.startswith('appCat.') or col in ['screen', 'call', 'sms']]
for col in duration_vars:
    rules[col] = lambda x: x < 0

# Check each rule
print("Unrealistic values found:")
for var, rule in rules.items():
    if var in daily_df_median_full.columns:
        mask = rule(daily_df_median_full[var])
        n_unrealistic = mask.sum()
        if n_unrealistic > 0:
            print(f"{var}: {n_unrealistic} values")


##### Method 2: Kalman
Using kalman for non-binary and non-duration and use the median method for other variables

In [ ]:
# variables we will do kalman on
kalman_vars = ['mood', 'circumplex.arousal', 'circumplex.valence', 'activity']

# kalman smoothing
def kalman_impute(series):
    if series.isnull().all(): # if all values are nan, keep nans dont do kalman
        return series

        model = UnobservedComponents(series, level='local level') #kalman
        result = model.fit(disp=False) # fit the model
        smoothed = result.smoothed_state[0] # get smoothed values from the fitted model
        series[series.isna()] = smoothed[series.isna()] # replace the missing values with the smoothed values
    return series

for var in kalman_vars:
    daily_df_kalman_full[var] = (
        daily_df_kalman_full.groupby('id')[var]
        .transform(kalman_impute)
    )

# after kalman apply fallback ffill + bfill for any still-missing kalman_vars
for var in kalman_vars:
    daily_df_kalman_full[var] = (
        daily_df_kalman_full.groupby('id')[var]
        .apply(lambda x: x.ffill().bfill())
        .reset_index(level=0, drop=True)
    )


In [ ]:
non_kalman_vars = [col for col in daily_df_kalman_full.columns if col not in kalman_vars + ['id', 'date']]

for var in non_kalman_vars:
    prev = daily_df_kalman_full.groupby('id')[var].shift(1)
    next_ = daily_df_kalman_full.groupby('id')[var].shift(-1)
    interpolated = (prev + next_) / 2
    daily_df_kalman_full[var] = daily_df_kalman_full[var].fillna(interpolated)

    daily_df_kalman_full[var] = (
        daily_df_kalman_full.groupby('id')[var]
        .apply(lambda x: x.ffill().bfill())
        .reset_index(level=0, drop=True)
    )

daily_df_kalman_full.head()

In [ ]:
missing_values_kalman = daily_df_kalman_full.isna().sum()
missing_values_kalman

In [ ]:
# Define rules per variable
rules = {
    'mood': lambda x: (x < 1) | (x > 10),
    'circumplex.arousal': lambda x: (x < -2) | (x > 2),
    'circumplex.valence': lambda x: (x < -2) | (x > 2),
    'activity': lambda x: (x < 0) | (x > 1),
}

# Add all duration-like columns
duration_vars = [col for col in daily_df_kalman_full.columns if col.startswith('appCat.') or col in ['screen', 'call', 'sms']]
for col in duration_vars:
    rules[col] = lambda x: x < 0

# Check each rule
print("🔍 Unrealistic values found:")
for var, rule in rules.items():
    if var in daily_df_kalman_full.columns:
        mask = rule(daily_df_kalman_full[var])
        n_unrealistic = mask.sum()
        if n_unrealistic > 0:
            print(f"{var}: {n_unrealistic} values")


#### Trimming first 5 days (for prediction dataset)

In [ ]:
# full version for training
daily_df_median_all = daily_df_median_full.copy()
daily_df_kalman_all = daily_df_kalman_full.copy()

# trimming first 5 days, for prediction dataset
# median method set
daily_df_median_predict = daily_df_median_all.sort_values(['id', 'date']).copy()
daily_df_median_predict['day_index'] = daily_df_median_predict.groupby('id').cumcount()
daily_df_median_predict = daily_df_median_predict[daily_df_median_predict['day_index'] >= 5].drop(columns='day_index')

# kalman method set
daily_df_kalman_predict = daily_df_kalman_all.sort_values(['id', 'date']).copy()
daily_df_kalman_predict['day_index'] = daily_df_kalman_predict.groupby('id').cumcount()
daily_df_kalman_predict = daily_df_kalman_predict[daily_df_kalman_predict['day_index'] >= 5].drop(columns='day_index')

In [ ]:
print("After trimming first 5 days:")
print("median shape:", daily_df_median_predict.shape)
print("kalman shape:", daily_df_kalman_predict.shape)

27 users × 5 days = 135 rows dropped